# **Setup Environment**

In [ ]:
import sys
import os
import numpy as np
from tqdm import tqdm

sys.path.append(os.path.abspath(".."))

from gymnasium.vector import SyncVectorEnv
from core.env.core import SnakeEnv
from core.env.types import ObserveType

num_envs, total_episodes = 16, 100000
env = SyncVectorEnv(
    [
        lambda i=i: SnakeEnv(
            width=20,
            height=20,
            obs_type=ObserveType.VEC_11,
            num_apples=3,
            num_obstacles=15,
            seed=42 + i,
            reward_options={
                "reward_apple": 5.0,
                "reward_step": -0.01,
                "reward_loop_penalty": -0.1,
                "reward_death_wall": -20.0,
                "reward_death_self": -20.0,
                "reward_shaping_closer": 0.3,
                "reward_shaping_further": -0.1,
                "reward_complete": 100.0,
            },
        )
        for i in range(num_envs)
    ]
)

epsilon_decay = (0.01) ** (1 / (total_episodes * 0.6))

training_logs = []
episode_rewards = np.zeros(num_envs)
completed = 0
states, infos = env.reset()
best_reward = -np.inf

Parallel Training:   5%|▌         | 5134/100000 [00:05<01:39, 956.02it/s] 

Ep 5000/100000 | Avg Reward (last 100): -14.78 | Eps: 0.681 | Best Avg: -13.42


Parallel Training:  10%|█         | 10078/100000 [00:10<01:51, 806.21it/s]

Ep 10000/100000 | Avg Reward (last 100): -11.15 | Eps: 0.464 | Best Avg: -9.04


Parallel Training:  15%|█▌        | 15089/100000 [00:17<02:00, 702.81it/s]

Ep 15000/100000 | Avg Reward (last 100): -5.98 | Eps: 0.316 | Best Avg: -2.58


Parallel Training:  16%|█▌        | 16177/100000 [00:19<01:39, 844.49it/s]


KeyboardInterrupt: 

---

# **Initialize Agent**

In [ ]:
from agents.q_learning import QLearningAgent

agent_name = "q_learning_snake_2"
agent = QLearningAgent(
    state_dim=2048,
    action_dim=3,
    lr=0.05,
    gamma=0.99,
    epsilon_decay=epsilon_decay,
    seed=42,
)

---

# **Train Agent**

In [ ]:
with tqdm(total=total_episodes, desc="Parallel Training") as pbar:
    while completed < total_episodes:
        actions = [int(agent.act(s)) for s in states]
        next_states, rewards, terminated, truncated, next_infos = env.step(actions)
        for i in range(len(actions)):
            s = states[i]
            a = actions[i]
            r = float(rewards[i])
            ns = next_states[i]
            done_i = bool(terminated[i])
            trunc_i = bool(truncated[i])

            agent.update(s, a, r, ns, done_i)
            episode_rewards[i] += r

            if done_i or trunc_i:
                completed += 1
                if completed <= total_episodes:
                    pbar.update(1)
                    agent.train()
                    training_logs.append(
                        {"episode": completed, "reward": float(episode_rewards[i]), "epsilon": agent.epsilon}
                    )

                    if len(training_logs) >= 100:
                        recent_avg = np.mean([log["reward"] for log in training_logs[-100:]])
                        if recent_avg > best_reward:
                            best_reward = recent_avg
                            agent.save(f"{agent_name}_best.pkl")

                    if completed % 5000 == 0:
                        recent_avg = (
                            np.mean([log["reward"] for log in training_logs[-100:]])
                            if len(training_logs) >= 100
                            else float(episode_rewards[i])
                        )
                        tqdm.write(
                            f"Ep {completed}/{total_episodes} | Avg Reward (last 100): {recent_avg:.2f} | Eps: {agent.epsilon:.3f} | Best Avg: {best_reward:.2f}"
                        )
                episode_rewards[i] = 0.0

        states = next_states

env.close()

## **Save Last Model**

In [ ]:
agent.save(f"{agent_name}.pkl")

---

# **Evaluate Agent**

In [ ]:
from core.utils import save_metrics

save_metrics(training_logs, f"{agent_name}_training_logs.csv")

In [4]:
from core.utils import evaluate_agent

evaluate_agent(agent, seed=67)

Evaluating Agent: 100%|██████████| 100/100 [00:02<00:00, 35.53it/s]


Metric          | Average  | Max      | Std Dev 
------------------------------------------------------------
Rewards         | 226.49   | 487.78   | 104.57  
Apples          | 22.43    | 50.00    | 10.01   
Steps           | 329.99   | 864.00   | 163.66  

Death Distribution:
 - self: 87 (87.0%)
 - wall: 13 (13.0%)



({'avg': 226.49129999999977,
  'max': 487.77999999999497,
  'sd': 104.56795798575108},
 {'avg': 22.43, 'max': 50.0, 'sd': 10.009250721207858},
 {'avg': 329.99, 'max': 864.0, 'sd': 163.66053250554944},
 {<DeathReason.SELF: 'self'>: 87, <DeathReason.WALL: 'wall'>: 13})

---